In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import os
from pathlib import Path
from typing import List, Dict, Any, Optional
from tqdm.auto import tqdm
from pypdf import PdfReader
from dotenv import load_dotenv

# Langhchain framework
from langchain_core.documents import Document
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# HF + Groq clients
from huggingface_hub import InferenceClient
from groq import Groq

import warnings
warnings.filterwarnings("ignore")

## Load API keys for credential key

In [2]:
ENV_PATH = Path.cwd().parent.parent / "credit_risk_production" / ".env"
load_dotenv(dotenv_path=ENV_PATH)

HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(f"HF token loaded:  {HUGGINGFACE_API_KEY[:3]}...")
print(f"GROQ token loaded: {GROQ_API_KEY[:3]}...")

HF token loaded:  hf_...
GROQ token loaded: gsk...


## Load database

In [3]:
DATA = Path.cwd().parent.parent / "credit_risk_production" / "database" / "data" / "merged_credit_risk_data.parquet"
FEATURES = Path.cwd().parent.parent / "credit_risk_production" / "database" / "data" / "features_data.parquet"

df = pd.read_parquet(DATA)
features = pd.read_parquet(FEATURES)

print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Features loaded: {features.shape[0]} rows, {features.shape[1]} columns")

Data loaded: 51336 rows, 87 columns
Features loaded: 51336 rows, 47 columns


## Load Fraud ML Models

In [4]:
# Load ML models from bundle
MODEL_BUNDLE = Path.cwd().parent.parent / "data_science" / "models" / "fraud_models" / "model_bundle.joblib"
with open(MODEL_BUNDLE, "rb") as f:
    model_bundle = joblib.load(f)
    print(f"Model bundle loaded: {list(model_bundle.keys())}")

# Load best parameters for each model
PARAMS = Path.cwd().parent.parent / "data_science" / "models" / "metrics" / "best_parameters.json"
with open(PARAMS, "r") as f:
    best_params = json.load(f)
    print(f"Best parameters loaded: {list(best_params.keys())}")

# Load metrics model
METRICS = Path.cwd().parent.parent / "data_science" / "models" / "metrics" / "model_metrics.csv"
metrics_df = pd.read_csv(METRICS)
display(metrics_df)

Model bundle loaded: ['models', 'scaler', 'label_encoders', 'feature_columns', 'class_labels']
Best parameters loaded: ['Logistic Regression', 'Random Forest', 'Decision Tree', 'XGBoost', 'K-Nearest Neighbors']


,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.957148,0.955984,0.988137,0.971795,0.958286
1,Random Forest,0.982859,0.986372,0.990744,0.988554,0.998319
2,Decision Tree,0.986560,0.995266,0.986703,0.990966,0.993473
3,XGBoost,0.987924,0.998547,0.985269,0.991864,0.998564
4,K-Nearest Neighbors,0.869108,0.885370,0.947464,0.915365,0.906852


In [5]:
model_bundle['models']

{'Logistic Regression': LogisticRegression(C=10, random_state=42),
 'Random Forest': RandomForestClassifier(bootstrap=False, min_samples_leaf=2,
                        min_samples_split=10, random_state=42),
 'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_leaf=4, min_samples_split=5,
                        random_state=42),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=1.0, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               gamma=None, grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.01, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=5, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
        

In [6]:
best_params

{'Logistic Regression': {'solver': 'lbfgs', 'penalty': 'l2', 'C': 10},
 'Random Forest': {'n_estimators': 100,
  'min_samples_split': 10,
  'min_samples_leaf': 2,
  'max_depth': None,
  'bootstrap': False},
 'Decision Tree': {'min_samples_split': 5,
  'min_samples_leaf': 4,
  'max_depth': 10},
 'XGBoost': {'n_estimators': 200,
  'max_depth': 5,
  'learning_rate': 0.01,
  'colsample_bytree': 1.0},
 'K-Nearest Neighbors': {'weights': 'distance',
  'n_neighbors': 9,
  'metric': 'manhattan'}}

# 📚 Document Ingestion — PDF → Chunks (per-document strategy)

In [7]:
PDF_DIR = Path("../../credit_risk_production/database/pdf")

pdf_files = {
    "delinquency":  PDF_DIR / "Delinquency_Classification .pdf",
    "fraud":        PDF_DIR / "Fraud_Typologies_and_Red Flags .pdf",
    "regulatory":   PDF_DIR / "Regulatory_Risk Policy Core .pdf",
    "scorecard":    PDF_DIR / "Scorecard_Cut-off Policy.pdf"
}

raw_texts: Dict[str, str] = {}

for name, path in pdf_files.items():
    reader = PdfReader(str(path))
    text = "\n".join((page.extract_text() or "") for page in reader.pages)
    raw_texts[name] = text
    print(f"{name:12s} | pages={len(reader.pages):3d} | chars={len(text):,}")

delinquency  | pages= 93 | chars=191,238
fraud        | pages=  9 | chars=25,113
regulatory   | pages= 11 | chars=34,419
scorecard    | pages=178 | chars=465,508


## Chunking strategy to retrieve Document augmentation for each PDFs to enrich vocab on LLM
- #### Strategy 1: Delinquency Classification — rule-based chunking
- #### Strategy 2: Fraud Typologies — focusing on fraud paragraph to be chunked
- #### Strategy 3: Regulatory Policy — recursive with heading preservation
- #### Strategy 4: Scorecard Cut-off — table-aware chunking
- #### Final Strategy: Combine all chunks

In [8]:
def chunk_delinquency(text: str) -> List[Document]:
    """Split by classification headings, falls back to paragraph split."""
    keywords = ["Standard", "Sub-standard", "Substandrad", "Doubtful", "Loss"]

    # Crude split: find the index of each keyword and slice
    positions = []
    for kw in keywords:
        idx = text.lower().find(kw.lower())
        if idx != -1:
            positions.append((idx, kw))
    positions.sort()

    docs: List[Document] = []
    if not positions:

        splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
        for i, chunk in enumerate(splitter.split_text(text)):
            docs.append(Document(
                page_content=chunk,
                metadata={"doc_type": "delinquency", "chunk_id": i, "class_level": "unknown"}
            ))
        return docs

    for i, (start, kw) in enumerate(positions):
        end = positions[i + 1][0] if i + 1 < len(positions) else len(text)
        chunk = text[start:end].strip()

        if len(chunk) < 30:
            continue

        docs.append(Document(
            page_content=chunk,
            metadata={"doc_type": "delinquency", "class_level": kw.lower(), "chunk_id": i},
        ))

    return docs

# Usage on delinquency PDF
deling_docs = chunk_delinquency(raw_texts["delinquency"])
print(f"Delinquency chunks: {len(deling_docs)}")
print(deling_docs[3].page_content[:300], "\n--")

Delinquency chunks: 4
sub-standard' immediately on restructuring, 
all borrowers, with the exception of the borrowal categories specified in para 14.1 below ( i.e 
consumer and personal advances, advances classified as capital market and real estate 
exposures), will be entitled to retain the asset classification upon re 
--


#### Strategy 2: Fraud Typologies — focusing on fraud paragraph to be chunked

In [13]:
def chunk_fraud(text: str) -> List[Document]:
    """Split by typology headings, falls back to paragraph split."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800, 
        chunk_overlap=80,
        separators=["\n\n\n", "\n\n", "\n", ". ", " "]
    )
    docs = []
    for i, chunk in enumerate(splitter.split_text(text)):
        docs.append(Document(
            page_content=chunk,
            metadata={"doc_type": "fraud", "chunk_id": i, "typology": "unspecified"}
        ))
    return docs

# Usage on fraud PDF
fraud_docs = chunk_fraud(raw_texts["fraud"])
print(f"Fraud chunks: {len(fraud_docs)}")
print(fraud_docs[3].page_content[:300], "\n--")

Fraud chunks: 36
platforms achieved a 61% improvement in detection accuracy, 48% reduction in false positives, and 72% faster 
investigation turnaround. The project management model introduced in this article outlines the lifecycle for planning, 
building, validating, deploying, and governing these systems, emphasiz 
--


### Strategy 3: Regulatory Policy — recursive with heading preservation

In [14]:
def chunk_regulatory(text: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800, 
        chunk_overlap=120,
        separators=["\n\n", "\n", ". ", " "]
    )
    docs = []
    for i, chunk in enumerate(splitter.split_text(text)):
        docs.append(Document(
            page_content=chunk,
            metadata={"doc_type": "regulatory", "chunk_id": i},
        ))
    return docs

# Usage on regulatory PDF
regulatory_docs = chunk_regulatory(raw_texts["regulatory"])
print(f"Regulatory chunks: {len(regulatory_docs)}")
print(regulatory_docs[1].page_content[:411], "\n--")

Regulatory chunks: 53
(FPC). However, despite these guidelines, rising consumer complaints indicate a gap between regulatory 
expectations and actual practices. This study aims to empirically examine the compliance of FPC norms among 
Banks and Housing Finance Companies (HFCs), as perceived by lending officials and borrowers. Primary data 
were collected from 294 borrowers and 102 lending branches using structured questionnaires. 
--


### Strategy 4: Scorecard Cut-off — table-aware chunking

In [15]:
import re

def chunk_scorecard(text: str) -> List[Document]:
    """Detect numeric ranges like 700-750 or 700 - 750 and split accordingly."""
    pattern = re.compile(r"(\d{3})\s*[--to]+\s*(\d{3})")
    matches = list(pattern.finditer(text))

    docs: List[Document] = []
    if not matches:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=700, 
            chunk_overlap=80,
            separators=["\n\n", "\n", ". ", " "]
        )
        for i, chunk in enumerate(splitter.split_text(text)):
            docs.append(Document(
                page_content=chunk,
                metadata={"doc_type": "scorecard", "chunk_id": i, "min_score": None, "max_score": None}
            ))
        return docs

    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        chunk = text[start:end].strip()

        if len(chunk) < 30:
            continue
        docs.append(Document(
            page_content=chunk,
            metadata={
                "doc_type": "scorecard",
                "chunk_id": i,
                "min_score": int(m.group(1)),
                "max_score": int(m.group(2))
            }
        ))
    return docs

# Usage on scorecard PDF
score_docs = chunk_scorecard(raw_texts["scorecard"])
print(f"Scorecard chunks: {len(score_docs)}")
print(score_docs[1].page_content[:300], "\n--")

Scorecard chunks: 12
300 to 850.  
 
Credit bureau scores consider five general groups of predictive variables: 
– Previous performance, including the severity and frequency of poor performance and 
how recently the poor performance occurred. 
– Current level and use of nonmortgage debt. 
– Amount of time that credit ha 
--


### Combine all chunks

In [16]:
all_docs: List[Document] = deling_docs + fraud_docs + regulatory_docs + score_docs
print(f"Total chunks: {len(all_docs)}")
print(f"By type: {pd.Series([d.metadata['doc_type'] for d in all_docs]).value_counts().to_dict()}")

Total chunks: 105
By type: {'regulatory': 53, 'fraud': 36, 'scorecard': 12, 'delinquency': 4}


## 🧩 Customer-Row → Narrative Document

### Build feature groups

In [17]:
FEATURE_GROUPS = {
    "trade_lines": [
        "Total_TL", "Tot_Closed_TL", "Tot_Active_TL",
        "Total_TL_opened_L6M", "Tot_TL_closed_L6M",
        "pct_tl_open_L6M", "pct_tl_closed_L6M",
        "pct_active_tl", "pct_closed_tl",
        "Total_TL_opened_L12M", "Tot_TL_closed_L12M",
        "pct_tl_open_L12M", "pct_tl_closed_L12M",
    ],
    "product_mix": [
        "Auto_TL", "CC_TL", "Consumer_TL", "Gold_TL",
        "Home_TL", "PL_TL", "Secured_TL", "Unsecured_TL", "Other_TL",
    ],
    "payment_behavior": ["Tot_Missed_Pmnt", "Age_Oldest_TL", "Age_Newest_TL"],
    "delinquency": [
        "time_since_recent_payment", "time_since_first_deliquency",
        "time_since_recent_deliquency", "num_times_delinquent",
        "max_delinquency_level", "max_recent_level_of_deliq",
        "num_deliq_6mts", "num_deliq_12mts", "num_deliq_6_12mts",
        "max_deliq_6mts", "max_deliq_12mts",
        "num_times_30p_dpd", "num_times_60p_dpd",
        "num_std", "num_std_6mts", "num_std_12mts",
        "num_sub", "num_sub_6mts", "num_sub_12mts",
        "num_dbt", "num_dbt_6mts", "num_dbt_12mts",
        "num_lss", "num_lss_6mts", "num_lss_12mts",
        "recent_level_of_deliq",
    ],
    "enquiries": [
        "tot_enq", "CC_enq", "CC_enq_L6m", "CC_enq_L12m",
        "PL_enq", "PL_enq_L6m", "PL_enq_L12m",
        "time_since_recent_enq", "enq_L12m", "enq_L6m", "enq_L3m",
    ],
    "demographics": ["MARITALSTATUS"],
}

### Derived features

In [44]:
def add_derived_features(row: pd.DataFrame) -> pd.Series:
    """Add derived features to a row based on existing features."""
    r = row.copy()
    def div(a, b):
        return float(a) / float(b) if pd.notna(a) and pd.notna(b) and b not in (0, None) else 0.0

    r["delinq_velocity"]   = div(row.get("num_deliq_6mts", 0), (row.get("num_deliq_12mts", 0) or 0) + 1)
    r["enq_velocity"]      = div(row.get("enq_L3m", 0), (row.get("enq_L12m", 0) or 0) + 1)
    r["unsecured_ratio"]   = div(row.get("Unsecured_TL", 0), (row.get("Total_TL", 0) or 0) + 1)
    r["active_ratio"]      = div(row.get("Tot_Active_TL", 0), (row.get("Total_TL", 0) or 0) + 1)
    r["recent_open_ratio"] = div(row.get("Total_TL_opened_L6M", 0), (row.get("Total_TL", 0) or 0) + 1)
    r["dpd_score"]         = float(row.get("num_times_30p_dpd", 0) or 0) * 1 + float(row.get("num_times_60p_dpd", 0) or 0) * 2
    r["asset_class_score"] = (
        float(row.get("num_sub", 0) or 0) * 1
        + float(row.get("num_dbt", 0) or 0) * 2
        + float(row.get("num_lss", 0) or 0) * 3
    )
    return r

df_enriched = df.apply(add_derived_features, axis=1)
print("\nAdded derived features to the dataset.")
df_enriched[["delinq_velocity","enq_velocity","unsecured_ratio","dpd_score","asset_class_score", "active_ratio", "recent_open_ratio"]].head(3)


Added derived features to the dataset.


,delinq_velocity,enq_velocity,unsecured_ratio,dpd_score,asset_class_score,active_ratio,recent_open_ratio
0,0.0,0.0,0.666667,0.0,0.0,0.166667,0.000000
1,0.0,0.0,0.500000,0.0,0.0,0.500000,0.000000
2,0.1,0.0,0.666667,0.0,0.0,0.888889,0.111111


## Convert each row to a narrative text chunk

In [19]:
def row_to_narrative(row: pd.Series, customer_id: Any = None) -> str:
    """Convert a customer row to a narrative string."""
    parts = [f"CUSTOMER_PROFILE id={customer_id}"]

    for group, cols in FEATURE_GROUPS.items():
        lines = [f" {c}={row[c]}" for c in cols if c in row.index and pd.notna(row[c])]
        if lines:
            parts.append(f"[{group.upper()}]\n" + "\n".join(lines))

    derived_cols = ["delinq_velocity", "enq_velocity", "unsecured_ratio",
                    "active_ratio", "recent_open_ratio", "dpd_score", "asset_class_score"]
    dl = [f" {c}={row[c]:.4f}" for c in derived_cols if c in row.index and pd.notna(row[c])]
    if dl:
        parts.append("[DERIVED]\n" + "\n".join(dl))

    return "\n".join(parts)

# id column: use index if no explicit id column
id_col = "customer_id" if "customer_id" in df_enriched.columns else None

# Build function customer doc
def make_customer_doc(row: pd.Series) -> Document:
    cid = row[id_col] if id_col else row.name
    return Document(
        page_content=row_to_narrative(row, cid),
        metadata={"doc_type": "customer_profile", "customer_id": str(cid)}
    )

# Check usage
print(make_customer_doc(df_enriched.iloc[0]).page_content[:600])

CUSTOMER_PROFILE id=0
[TRADE_LINES]
 Total_TL=5
 Tot_Closed_TL=4
 Tot_Active_TL=1
 Total_TL_opened_L6M=0
 Tot_TL_closed_L6M=0
 pct_tl_open_L6M=0.0
 pct_tl_closed_L6M=0.0
 pct_active_tl=0.2
 pct_closed_tl=0.8
 Total_TL_opened_L12M=0
 Tot_TL_closed_L12M=0
 pct_tl_open_L12M=0.0
 pct_tl_closed_L12M=0.0
[PRODUCT_MIX]
 Auto_TL=0
 CC_TL=0
 Consumer_TL=0
 Gold_TL=1
 Home_TL=0
 PL_TL=4
 Secured_TL=1
 Unsecured_TL=4
 Other_TL=0
[PAYMENT_BEHAVIOR]
 Tot_Missed_Pmnt=0
 Age_Oldest_TL=72
 Age_Newest_TL=18
[DELINQUENCY]
 time_since_recent_payment=549
 time_since_first_deliquency=35
 time_since_recent_deliquen


In [30]:
# Build customer documents
SAMPLE_N = 3000
df_sample = df_enriched.sample(n=SAMPLE_N, random_state=42)

customer_docs = [make_customer_doc(r) for _, r in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Building customer docs")]
print(f"Customer docs sample: {len(customer_docs)}")
print("Customer doc sample:")
print(customer_docs[:1])

Building customer docs:   0%|          | 0/3000 [00:00<?, ?it/s]

Customer docs sample: 3000
Customer doc sample:
[Document(metadata={'doc_type': 'customer_profile', 'customer_id': '8564'}, page_content='CUSTOMER_PROFILE id=8564\n[TRADE_LINES]\n Total_TL=3\n Tot_Closed_TL=1\n Tot_Active_TL=2\n Total_TL_opened_L6M=0\n Tot_TL_closed_L6M=0\n pct_tl_open_L6M=0.0\n pct_tl_closed_L6M=0.0\n pct_active_tl=0.667\n pct_closed_tl=0.333\n Total_TL_opened_L12M=0\n Tot_TL_closed_L12M=0\n pct_tl_open_L12M=0.0\n pct_tl_closed_L12M=0.0\n[PRODUCT_MIX]\n Auto_TL=1\n CC_TL=0\n Consumer_TL=0\n Gold_TL=0\n Home_TL=0\n PL_TL=0\n Secured_TL=2\n Unsecured_TL=1\n Other_TL=2\n[PAYMENT_BEHAVIOR]\n Tot_Missed_Pmnt=0\n Age_Oldest_TL=51\n Age_Newest_TL=30\n[DELINQUENCY]\n time_since_recent_payment=138\n time_since_first_deliquency=14\n time_since_recent_deliquency=12\n num_times_delinquent=2\n max_delinquency_level=26\n max_recent_level_of_deliq=26\n num_deliq_6mts=0\n num_deliq_12mts=0\n num_deliq_6_12mts=0\n max_deliq_6mts=0\n max_deliq_12mts=0\n num_times_30p_dpd=0\n num_times_

## Vector Store (ChromaDB + HuggingFace Embeddings)

In [36]:
EMBED_MODEL = "all-MiniLM-L6-v2" 

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={"normalize_embeddings": True}
)
print(f"Embeddings model: {EMBED_MODEL}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings model: all-MiniLM-L6-v2


## Persist collection to disk for ChromaDB

In [41]:
DATABASE_PATH_DIR = Path.cwd().parent.parent / "data_science" / "database" / "LLM"

PERSIST_DIR = DATABASE_PATH_DIR / "chroma_store"
PERSIST_DIR.mkdir(parents=True, exist_ok=True)

def build_load_collection(name: str, docs: List[Document]):
    """Store chromadb collection"""
    store = Chroma(
        collection_name=name,
        embedding_function=embeddings,
        persist_directory=str(PERSIST_DIR)
    )
    existing = store._collection.count()

    if existing == 0 and docs:
        BATCH = 256 
        for i in range(0, len(docs), BATCH):
            store.add_documents(docs[i:i+BATCH])
        store.persist()

    print(f"Collection '{name}': {store._collection.count()} vectors")
    return store

# Collection (customers + policies brief)
customer_store = build_load_collection("customer_profiles", customer_docs)
policy_docs = deling_docs + fraud_docs + regulatory_docs + score_docs
policy_store = build_load_collection("policy_documents", policy_docs)

print(f"Customer store: {customer_store._collection.count()} vectors")
print(f"Policy store: {policy_store._collection.count()} vectors")

Collection 'customer_profiles': 3000 vectors
Collection 'policy_documents': 105 vectors
Customer store: 3000 vectors
Policy store: 105 vectors


## 🤖 LLM Client with HF → Groq Fallback

In [42]:
from huggingface_hub import login
login(token=HUGGINGFACE_API_KEY)
print("✅ Logged in to Hugging Face Hub successfully.")

✅ Logged in to Hugging Face Hub successfully.


In [43]:
# Initialize HF and Groq clients
hf_client = InferenceClient(api_key=HUGGINGFACE_API_KEY) if HUGGINGFACE_API_KEY else None
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

def llm_chat(message: List[Dict[str, str]], max_tokens: int = 256, temperature: float = 0.3) -> Dict[str, Any]:
    """Try HF first, fallback to Groq if HF fails."""
    if hf_client:
        try:
            resp = hf_client.chat.completions.create(
                model="Qwen/Qwen2.5-Coder-32B-Instruct",
                messages=message,
                max_tokens=max_tokens,
                temperature=temperature
            )
            return {"text": resp.choices[0].message.content,
                    "provider": "huggingface"}
        except Exception as e:
            err_msg = str(e)
            if "402" in err_msg or "depleted" in err_msg:
                print("⚠️  HF quota depleted (402). Falling back to Groq...")
            else:
                print(f"❌  Hugging Face failed: {str(e)}")

    # Fallback to Groq
    if groq_client:
        try:
            resp = groq_client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=message,
                max_tokens=max_tokens,
                temperature=temperature
            )
            return {"text": resp.choices[0].message.content,
                    "provider": "groq"}
        except Exception as e:
            print(f"❌  Groq failed: {str(e)}")

    raise RuntimeError("⚠️ No LLM provider available. Please check your API keys.")

# Sanity test
print(llm_chat([
    {"role": "user",
     "content": "Reply with the single word: READY"}
]))

{'text': 'READY', 'provider': 'huggingface'}


## 🧠 RAG Pipeline

In [46]:
def retrieve_context(row: pd.Series, k_customers: int = 3, k_policies: int = 5) -> Dict[str, Any]:
    """Retrieve relevant policy and similar customer documents for a given row"""
    query = row_to_narrative(row)
    cust_hits = customer_store.similarity_search(query, k=k_customers)

    # Policy hits: bias query toward delinquency + enquiries since driver risk
    policy_query = (
        f"{query}\n"
        f"focus: delinquency, {row.get('num_deliq_12mts', 0)} misses in 12 months"
        f"unsecured ratio {row.get('unsecured_ratio', 0):.2f}"
        f"enquiries in last 12 months: {row.get('enq_L12m', 0)}"
        f"asset class score: {row.get('asset_class_score', 0):.2f}"
        f"credit card enquiries: {row.get('CC_enq_L12m', 0)}"
    )
    policy_hits = policy_store.similarity_search(policy_query, k=k_policies)

    return {"customers": cust_hits, "policies": policy_hits}

## Build prompt (system + context + task)

In [ ]:
SYSTEM_PROMPT = 